# 🏥 MediScan AI — Model Training Notebook
Train all three disease prediction models and download the `.pkl` files.


In [ ]:
# Install dependencies
!pip install scikit-learn pandas numpy joblib -q

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib, json, os

np.random.seed(42)
os.makedirs('dataset', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)
print('✅ Setup complete')

## 1️⃣ Diabetes Dataset Generation

In [ ]:
def generate_diabetes_data(n=1200):
    outcome = np.random.choice([0,1], size=n, p=[0.65,0.35])
    glucose = np.where(outcome, np.random.normal(140,25,n), np.random.normal(110,20,n))
    bmi     = np.where(outcome, np.random.normal(35,6,n),   np.random.normal(28,5,n))
    insulin = np.where(outcome, np.random.normal(180,80,n), np.random.normal(80,40,n))
    bp      = np.where(outcome, np.random.normal(80,12,n),  np.random.normal(70,10,n))
    skin    = np.where(outcome, np.random.normal(32,10,n),  np.random.normal(20,8,n))
    dpf     = np.where(outcome, np.random.uniform(0.4,2.0,n), np.random.uniform(0.1,0.8,n))
    preg    = np.where(outcome, np.random.randint(2,15,n),  np.random.randint(0,8,n))
    age     = np.where(outcome, np.random.randint(30,70,n), np.random.randint(20,60,n))
    return pd.DataFrame({
        'Pregnancies':np.clip(preg,0,17).astype(int),
        'Glucose':np.clip(glucose,50,200).astype(int),
        'BloodPressure':np.clip(bp,40,122).astype(int),
        'SkinThickness':np.clip(skin,0,99).astype(int),
        'Insulin':np.clip(insulin,0,846).astype(int),
        'BMI':np.round(np.clip(bmi,15,67),1),
        'DiabetesPedigreeFunction':np.round(np.clip(dpf,0.05,2.5),3),
        'Age':np.clip(age,18,81).astype(int),
        'Outcome':outcome
    })

df_d = generate_diabetes_data()
print('Diabetes shape:', df_d.shape)
df_d.head()

## 2️⃣ Heart Disease Dataset Generation

In [ ]:
def generate_heart_data(n=1200):
    outcome  = np.random.choice([0,1], size=n, p=[0.54,0.46])
    age      = np.where(outcome, np.random.randint(45,80,n), np.random.randint(30,65,n))
    sex      = np.random.choice([0,1], size=n, p=[0.32,0.68])
    cp       = np.where(outcome, np.random.choice([2,3],size=n), np.random.choice([0,1],size=n))
    trestbps = np.where(outcome, np.random.normal(145,22,n), np.random.normal(130,17,n))
    chol     = np.where(outcome, np.random.normal(260,55,n), np.random.normal(235,48,n))
    fbs      = np.where(outcome, np.random.choice([0,1],size=n,p=[0.55,0.45]), np.random.choice([0,1],size=n,p=[0.85,0.15]))
    restecg  = np.random.choice([0,1,2], size=n, p=[0.5,0.48,0.02])
    thalach  = np.where(outcome, np.random.normal(135,25,n), np.random.normal(158,22,n))
    exang    = np.where(outcome, np.random.choice([0,1],size=n,p=[0.35,0.65]), np.random.choice([0,1],size=n,p=[0.8,0.2]))
    oldpeak  = np.where(outcome, np.random.uniform(1.5,5.5,n), np.random.uniform(0,2.5,n))
    slope    = np.random.choice([0,1,2], size=n)
    ca       = np.where(outcome, np.random.choice([1,2,3],size=n), np.random.choice([0,1],size=n))
    thal     = np.where(outcome, np.random.choice([2,3],size=n), np.random.choice([1,2],size=n))
    return pd.DataFrame({
        'age':np.clip(age,25,80).astype(int), 'sex':sex, 'cp':cp,
        'trestbps':np.clip(trestbps,90,200).astype(int),
        'chol':np.clip(chol,120,600).astype(int), 'fbs':fbs, 'restecg':restecg,
        'thalach':np.clip(thalach,70,202).astype(int), 'exang':exang,
        'oldpeak':np.round(np.clip(oldpeak,0,6.2),1),
        'slope':slope, 'ca':np.clip(ca,0,4), 'thal':np.clip(thal,0,3), 'target':outcome
    })

df_h = generate_heart_data()
print('Heart shape:', df_h.shape)
df_h.head()

## 3️⃣ Parkinson's Dataset Generation

In [ ]:
def generate_parkinsons_data(n=900):
    outcome = np.random.choice([0,1], size=n, p=[0.25,0.75])
    fo      = np.where(outcome, np.random.normal(145,30,n), np.random.normal(195,35,n))
    fhi     = np.where(outcome, np.random.normal(180,40,n), np.random.normal(250,60,n))
    flo     = np.where(outcome, np.random.normal(110,25,n), np.random.normal(160,30,n))
    jitter  = np.where(outcome, np.random.uniform(0.003,0.015,n), np.random.uniform(0.001,0.006,n))
    shimmer = np.where(outcome, np.random.uniform(0.04,0.15,n),  np.random.uniform(0.01,0.06,n))
    nhr     = np.where(outcome, np.random.uniform(0.02,0.35,n),  np.random.uniform(0.005,0.02,n))
    hnr     = np.where(outcome, np.random.normal(18,5,n),        np.random.normal(24,4,n))
    rpde    = np.where(outcome, np.random.uniform(0.4,0.85,n),   np.random.uniform(0.25,0.55,n))
    dfa     = np.where(outcome, np.random.uniform(0.65,0.9,n),   np.random.uniform(0.55,0.75,n))
    s1      = np.where(outcome, np.random.uniform(-7.5,-4,n),    np.random.uniform(-8.5,-6,n))
    s2      = np.where(outcome, np.random.uniform(0.1,0.45,n),   np.random.uniform(0.05,0.2,n))
    d2      = np.where(outcome, np.random.uniform(2.2,3.5,n),    np.random.uniform(1.8,2.8,n))
    ppe     = np.where(outcome, np.random.uniform(0.15,0.55,n),  np.random.uniform(0.03,0.18,n))
    return pd.DataFrame({
        'MDVP_Fo_Hz':np.round(np.clip(fo,80,270),3),
        'MDVP_Fhi_Hz':np.round(np.clip(fhi,100,600),3),
        'MDVP_Flo_Hz':np.round(np.clip(flo,65,240),3),
        'MDVP_Jitter_pct':np.round(np.clip(jitter,0.001,0.03),5),
        'Shimmer':np.round(np.clip(shimmer,0.009,0.3),5),
        'NHR':np.round(np.clip(nhr,0.001,0.5),5),
        'HNR':np.round(np.clip(hnr,8,35),3),
        'RPDE':np.round(np.clip(rpde,0.2,0.95),6),
        'DFA':np.round(np.clip(dfa,0.5,0.97),6),
        'spread1':np.round(np.clip(s1,-9,-2),6),
        'spread2':np.round(np.clip(s2,0.02,0.55),6),
        'D2':np.round(np.clip(d2,1.5,4.0),6),
        'PPE':np.round(np.clip(ppe,0.02,0.7),6),
        'status':outcome
    })

df_p = generate_parkinsons_data()
print('Parkinsons shape:', df_p.shape)
df_p.head()

## 4️⃣ Train All Models & Save

In [ ]:
def train_and_save(name, df, target_col, feature_cols):
    print(f'\n=== Training: {name} ===')
    X = df[feature_cols]
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Save CSVs
    pd.concat([X_train,y_train],axis=1).to_csv(f'dataset/{name}_train.csv', index=False)
    pd.concat([X_test,y_test],axis=1).to_csv(f'dataset/{name}_test.csv', index=False)
    df.to_csv(f'dataset/{name}_full.csv', index=False)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    model = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
    model.fit(X_train_s, y_train)

    acc = accuracy_score(y_test, model.predict(X_test_s))
    print(classification_report(y_test, model.predict(X_test_s), target_names=['Negative','Positive']))

    joblib.dump(model,  f'saved_models/{name}_model.pkl')
    joblib.dump(scaler, f'saved_models/{name}_scaler.pkl')
    with open(f'saved_models/{name}_metrics.json','w') as f:
        json.dump({'accuracy':round(acc,4),'n_samples':len(df),'n_train':len(X_train),
                   'n_test':len(X_test),'features':feature_cols}, f, indent=2)
    print(f'✅ Saved — Accuracy: {acc*100:.2f}%')
    return acc

train_and_save('diabetes',   df_d, 'Outcome', ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age'])
train_and_save('heart',      df_h, 'target',  ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'])
train_and_save('parkinsons', df_p, 'status',  ['MDVP_Fo_Hz','MDVP_Fhi_Hz','MDVP_Flo_Hz','MDVP_Jitter_pct','Shimmer','NHR','HNR','RPDE','DFA','spread1','spread2','D2','PPE'])

print('\n🎉 ALL MODELS TRAINED & SAVED!')

## 5️⃣ Download All Files (Colab only)

In [ ]:
# Run this cell in Google Colab to download the trained models
try:
    from google.colab import files
    import glob
    for f in glob.glob('saved_models/*.pkl') + glob.glob('saved_models/*.json'):
        files.download(f)
    for f in glob.glob('dataset/*.csv'):
        files.download(f)
    print('All files downloaded!')
except ImportError:
    print('Not running in Colab — files are saved locally in saved_models/ and dataset/ folders.')